<img src="https://raw.githubusercontent.com/FrontierDevelopmentLab/2024-HL-GeoCL/main/public/FDL_black.png" width="200" alt="FDL logo" />

# SHEATH Inference Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrontierDevelopmentLab/2024-HL-GeoCL/blob/main/public/sheath_inference_quickstart.ipynb)

This notebook demonstrates how to run inference with a trained **SHEATH** model.

## What is SHEATH?

**SHEATH** (Solar wind High-speed Enhancements And Transients Handler) is a neural network that predicts solar wind parameters from features extracted from SDO (Solar Dynamics Observatory) imagery. It is part of the [GEO-CLoak](https://github.com/FrontierDevelopmentLab/2024-HL-GeoCL) pipeline developed during FDL-X Heliolab 2024.

### Inputs (26 features)
The model takes 26 features derived from segmented SDO/AIA images:
- **Coronal Hole (CH) and Active Region (AR) areas** from image segmentation
- **Total emission** through CH/AR masks for 10 AIA EUV/UV channels (in Angstroms: 131, 1600, 1700, 171, 193, 211, 304, 335, 94) 
- **Magnetic field components** (Bx, By, Bz) through CH/AR masks from HMI magnetograms

### Outputs (14 predictions)
The model predicts the **mean and standard deviation** of 7 solar wind quantities:

| Output | Unit | Description |
|--------|------|-------------|
| Speed | km/s | Solar wind bulk speed |
| Density | cm^-3 | Proton number density |
| Temperature | K | Proton temperature |
| Bt | nT | Total magnetic field magnitude |
| Bx | nT | Magnetic field GSM x-component |
| By | nT | Magnetic field GSM y-component |
| Bz | nT | Magnetic field GSM z-component |

The standard deviation outputs (Speedsd, Densitysd, ...) quantify the model's uncertainty.

### Architecture

SHEATH is a simple 3-layer MLP:

```
Input (26) --> Linear(256) --> ReLU --> Dropout
           --> Linear(256) --> ReLU --> Dropout
           --> Linear(14)  --> Output
```

## 1. Setup

Install the required dependencies. On Colab, `torch` is pre-installed; we just need `scikit-learn`.

In [ ]:
# Install dependencies (only needed on Colab or fresh environments)
# torch, numpy, and pandas are pre-installed on Colab
%pip install -q scikit-learn

In [ ]:
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler, StandardScaler

## 2. Load model artifacts from AWS S3

The model artifacts are hosted on a **public S3 bucket** (no credentials needed). We stream them directly into memory — nothing is saved to disk.

We need:
- **Model checkpoint** (`sheath_latest.pth`) — the trained weights
- **Input scaler** (`scaler_inputs.json`) — StandardScaler fitted on the training features
- **Target scaler** (`scaler_targets.json`) — MinMaxScaler fitted on the training targets
- **Sample data** (`sample_input_sheath.csv`) — 100 rows of real input data to try

In [ ]:
import io

S3_BASE = "https://nasa-radiant-data.s3.amazonaws.com/helioai-datasets/hl-geo/models"

# Stream model weights into memory
print("Streaming sheath_latest.pth...")
with urllib.request.urlopen(f"{S3_BASE}/sheath_latest.pth") as resp:
    model_bytes = io.BytesIO(resp.read())
print(f"  Loaded {model_bytes.getbuffer().nbytes:,} bytes")

# Stream scaler JSONs into memory
print("Streaming scaler_inputs.json...")
with urllib.request.urlopen(f"{S3_BASE}/examples/scaler_inputs.json") as resp:
    scaler_inputs_data = json.load(resp)

print("Streaming scaler_targets.json...")
with urllib.request.urlopen(f"{S3_BASE}/examples/scaler_targets.json") as resp:
    scaler_targets_data = json.load(resp)

# Stream sample CSV directly into a DataFrame
print("Streaming sample_input_sheath.csv...")
input_data = pd.read_csv(f"{S3_BASE}/examples/sample_input_sheath.csv")

print("\nAll artifacts loaded into memory.")

## 3. Define the model

The SHEATH model is a straightforward 3-layer MLP with ReLU activations and dropout for regularization. We reproduce it here so this notebook is fully self-contained, however the model is contained in the repository code. 

In [ ]:
class SHEATH_MLP(nn.Module):
    """3-layer MLP for solar wind parameter prediction."""

    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.fc3(x)
        return x


# Model hyperparameters (must match the training configuration)
INPUT_DIM = 26
HIDDEN_DIM = 256
OUTPUT_DIM = 14
DROPOUT_RATE = 0.5

print(f"Model: {INPUT_DIM} inputs -> {HIDDEN_DIM} hidden -> {OUTPUT_DIM} outputs")
total_params = sum(
    p.numel() for p in SHEATH_MLP(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).parameters()
)
print(f"Total parameters: {total_params:,}")

## 4. Load the trained model

We load the checkpoint, put the model in evaluation mode (this disables dropout), and select the appropriate device.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SHEATH_MLP(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    dropout_rate=DROPOUT_RATE,
)

state_dict = torch.load(model_bytes, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print("Model loaded successfully.")

## 5. Load the scalers

The model was trained on normalized data, so we need the same scalers used during training:

- **Input scaler** (StandardScaler): zero-mean, unit-variance normalization of the 26 input features
- **Target scaler** (MinMaxScaler): maps the 14 outputs to [0, 1] range during training; we invert this to get physical units

In [ ]:
def build_standard_scaler(d):
    """Reconstruct a StandardScaler from a dict."""
    scaler = StandardScaler()
    scaler.mean_ = np.array(d["mean"])
    scaler.scale_ = np.array(d["scale"])
    scaler.var_ = np.array(d["var"])
    scaler.n_features_in_ = d["n_features"]
    scaler.feature_names_in_ = np.array(d["feature_names"])
    return scaler


def build_minmax_scaler(d):
    """Reconstruct a MinMaxScaler from a dict."""
    scaler = MinMaxScaler(feature_range=tuple(d["feature_range"]))
    scaler.min_ = np.array(d["min"])
    scaler.scale_ = np.array(d["scale"])
    scaler.data_min_ = np.array(d["data_min"])
    scaler.data_max_ = np.array(d["data_max"])
    scaler.n_features_in_ = d["n_features"]
    scaler.feature_names_in_ = np.array(d["feature_names"])
    return scaler


scaler_inputs = build_standard_scaler(scaler_inputs_data)
scaler_targets = build_minmax_scaler(scaler_targets_data)

print(f"Input scaler: {scaler_inputs.n_features_in_} features")
print(f"  Feature names: {list(scaler_inputs.feature_names_in_)}")
print(f"\nTarget scaler: {scaler_targets.n_features_in_} targets")
print(f"  Target names: {list(scaler_targets.feature_names_in_)}")

## 6. Load input data

The input CSV should have 26 columns corresponding to SDO image features. Let's load the sample data and inspect it.

In [ ]:
# input_data was already streamed from S3 in step 2
print(f"Input shape: {input_data.shape}")
print(f"Columns: {input_data.columns.tolist()}")
print()
input_data.head()

The features fall into three groups:

| Prefix | Source | Meaning |
|--------|--------|---------|
| `CHArea`, `ARArea` | SDO/AIA segmentation | Fractional area of coronal holes / active regions |
| `CH___A`, `AR___A` | SDO/AIA channels | Total emission through CH/AR masks per wavelength |
| `CHB_`, `ARB_` | SDO/HMI magnetogram | Mean magnetic field (Bx, By, Bz) through CH/AR masks |

## 7. Run inference

The inference pipeline has three steps:
1. **Normalize** the input features using the training scaler
2. **Forward pass** through the model
3. **Inverse-transform** the outputs to get predictions in physical units

In [ ]:
# Step 1: Normalize inputs
features_scaled = scaler_inputs.transform(input_data.values)
print("Scaled input (first row):")
print(np.round(features_scaled[0], 3))

In [ ]:
# Step 2: Run the model
with torch.no_grad():
    x = torch.tensor(features_scaled, dtype=torch.float32, device=device)
    raw_output = model(x).cpu().numpy()

print(f"Raw model output shape: {raw_output.shape}")
print(f"Raw output (first row): {np.round(raw_output[0], 4)}")
print()
print("Note: these values are in the [0, 1] scaled space, not physical units yet.")

In [ ]:
# Step 3: Inverse-transform to physical units
predictions = scaler_targets.inverse_transform(raw_output)

TARGET_NAMES = [
    "Speed",
    "Density",
    "Temperature",
    "Bt",
    "Bx",
    "By",
    "Bz",
    "Speedsd",
    "Densitysd",
    "Temperaturesd",
    "Btsd",
    "Bxsd",
    "Bysd",
    "Bzsd",
]

results = pd.DataFrame(predictions, columns=TARGET_NAMES)
results.head()

## 8. Interpret the results

The 14 output columns are the **mean** and **standard deviation** of each solar wind parameter. Let's separate them and display them together.

In [ ]:
PARAM_NAMES = ["Speed", "Density", "Temperature", "Bt", "Bx", "By", "Bz"]
UNITS = ["km/s", "cm\u207b\u00b3", "K", "nT", "nT", "nT", "nT"]

for i, row in results.iterrows():
    if i >= 3:
        continue  # Limit to first 4 samples for display
    print(f"--- Sample {i + 1} ---")
    for name, unit in zip(PARAM_NAMES, UNITS):
        mean_val = row[name]
        std_val = row[f"{name}sd"]
        print(f"  {name:15s} = {mean_val:12.2f} \u00b1 {std_val:10.2f} {unit}")
    print()

## 9. Visualize predictions

Let's plot the predicted means with their uncertainty bands across our sample rows.

In [ ]:
fig, axes = plt.subplots(len(PARAM_NAMES), 1, figsize=(8, 14), sharex=True)

sample_indices = np.arange(len(results))

for ax, name, unit in zip(axes, PARAM_NAMES, UNITS):
    means = results[name].values
    stds = results[f"{name}sd"].values

    # Clip negative std values to zero for display
    stds = np.clip(stds, 0, None)

    ax.errorbar(sample_indices, means, yerr=stds, fmt="o-", capsize=4, markersize=6)
    ax.set_ylabel(f"{name} ({unit})")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample index")
axes[-1].set_xticks(sample_indices)
axes[0].set_title("SHEATH Predictions with Uncertainty")
plt.tight_layout()
plt.show()

## 10. Using your own data

To run inference on your own data, prepare a CSV with the 26 input features as columns:

```
CHArea,ARArea,CH131A,AR131A,CH1600A,AR1600A,CH1700A,AR1700A,CH171A,AR171A,
CH193A,AR193A,CH211A,AR211A,CH304A,AR304A,CH335A,AR335A,CH94A,AR94A,
CHBx,ARBx,CHBy,ARBy,CHBz,ARBz
```

These features are extracted from SDO/AIA images by segmenting coronal holes and active regions, then computing total emission per channel and mean magnetic field through each mask. See `geocloak/preprocess/sdoprep.py` for the full preprocessing pipeline.

Then run the same steps shown above:

```python
my_data = pd.read_csv("my_features.csv")
scaled = scaler_inputs.transform(my_data.values)

with torch.no_grad():
    x = torch.tensor(scaled, dtype=torch.float32, device=device)
    raw = model(x).cpu().numpy()

predictions = pd.DataFrame(
    scaler_targets.inverse_transform(raw),
    columns=TARGET_NAMES,
)
```

Alternatively, use the command-line script `inference_sheath.py` from the [GEO-CLoak repository](https://github.com/FrontierDevelopmentLab/2024-HL-GeoCL):

```bash
python inference_sheath.py \
    --input my_features.csv \
    --checkpoint sheath_best_model_checkpoint_1.pth \
    --scaler-inputs scaler_inputs.json \
    --scaler-targets scaler_targets.json \
    --output predictions.csv
```

## Acknowledgements

This work is the research product of FDL-X Heliolab, a public/private partnership between NASA, Trillium Technologies Inc (trillium.tech) and commercial AI partners Google Cloud, NVIDIA and Pasteur Labs & ISI, developing open science for all Humankind.